**Persiapan SparkSession**

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


**A. EXTRACT**

In [5]:
# Membaca data transaksi
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)

# Membaca data produk
df_produk = spark.read.json("tugas6_produk.json")

# Membaca data ulasan
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("Data transaksi:")
df_transaksi.printSchema()

print("Data produk:")
df_produk.printSchema()

print("Data ulasan:")
df_ulasan.printSchema()

Data transaksi:
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Data produk:
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Data ulasan:
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



**B. TRANSFORM**

1. Gabungkan transaksi dengan ulasan

In [6]:
df_transaksi_ulasan = df_transaksi.join(df_ulasan, on="order_id", how="left")
df_transaksi_ulasan.show(10)
print("jumlah transaksi setelah join ulasan:", df_transaksi_ulasan.count())

+--------+----------+------------+-------------------+------+
|order_id|product_id|unit_terjual|            tanggal|rating|
+--------+----------+------------+-------------------+------+
|     TX0|        21|           2|2026-10-10 00:00:00|     1|
|     TX1|         8|           6|2026-10-25 00:00:00|  NULL|
|     TX2|        25|           4|2026-10-13 00:00:00|     2|
|     TX3|         3|           4|2026-10-18 00:00:00|     5|
|     TX4|        19|           6|2026-10-07 00:00:00|  NULL|
|     TX5|        10|           3|2026-10-01 00:00:00|     5|
|     TX6|        26|           6|2026-10-24 00:00:00|     4|
|     TX7|         4|           1|2026-10-03 00:00:00|  NULL|
|     TX8|         7|           2|2026-10-09 00:00:00|     5|
|     TX9|        30|           1|2026-10-27 00:00:00|  NULL|
+--------+----------+------------+-------------------+------+
only showing top 10 rows



[Stage 18:>                                                         (0 + 1) / 1]

jumlah transaksi setelah join ulasan: 5000


2. Gabungkan dengan data produk

In [7]:
df_etl = df_transaksi_ulasan.join(df_produk, on="product_id", how="inner")
df_etl.show(10)
df_etl.printSchema()

+----------+--------+------------+-------------------+------+------+------------+-----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|
+----------+--------+------------+-------------------+------+------+------------+-----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|
|        26|     TX6|           6|2026-10-24 00:00:00|     4| 75000|   Kesehatan|  Produk-26|
|         4|     TX7|           1|2026-10-03 00:00:00|  NULL

3. Tambahkan total pendapatan sesuai soal: total_pendapatan = unit_terjual x harga

In [8]:
df_etl = df_etl.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))
df_etl.show(10)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|          225000|
|

**C. TRANSFORM**

In [18]:
from pyspark.sql.functions import col

df_etl = df_etl.fillna({"rating": 0})

df_etl = df_etl.withColumn(
    "ada_ulasan",
    col("rating") > 0
)

df_etl.show(10)

print(
    "Jumlah rating yang masih kosong:",
    df_etl.filter(col("rating").isNull()).count()
)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|ada_ulasan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|      true|
|         8|     TX1|           6|2026-10-25 00:00:00|     0|250000|     Fashion|   Produk-8|         1500000|     false|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|      true|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|      true|
|        19|     TX4|           6|2026-10-07 00:00:00|     0| 75000|Rumah Tangga|  Produk-19|          450000|     false|
|        10|     TX5|   

**D. LOAD**

In [15]:
!hdfs dfs -mkdir -p /user/kiaa/tugas6

In [21]:
df_etl.write \
    .mode("overwrite") \
    .partitionBy("kategori") \
    .parquet(
        "hdfs://localhost:9000/user/kiaa/tugas6/hasil_etl"
    )

In [22]:
!hdfs dfs -ls -R /user/kiaa/tugas6/hasil_etl

-rw-r--r--   3 kiaa supergroup          0 2026-09-24 09:22 /user/kiaa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - kiaa supergroup          0 2026-09-24 09:22 /user/kiaa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 kiaa supergroup      16366 2026-09-24 09:22 /user/kiaa/tugas6/hasil_etl/kategori=Elektronik/part-00000-100930a6-5019-4e32-8112-bf5e3d1b3844.c000.snappy.parquet
drwxr-xr-x   - kiaa supergroup          0 2026-09-24 09:22 /user/kiaa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 kiaa supergroup      11133 2026-09-24 09:22 /user/kiaa/tugas6/hasil_etl/kategori=Fashion/part-00000-100930a6-5019-4e32-8112-bf5e3d1b3844.c000.snappy.parquet
drwxr-xr-x   - kiaa supergroup          0 2026-09-24 09:22 /user/kiaa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 kiaa supergroup      10392 2026-09-24 09:22 /user/kiaa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-100930a6-5019-4e32-8112-bf5e3d1b3844.c000.snappy.parquet
drwxr-xr-x   - kiaa supergroup          0 2026-09-24 09:22 /user/k

In [23]:
df_final = spark.read.parquet(
    "hdfs://localhost:9000/user/kiaa/tugas6/hasil_etl"
)

In [24]:
print("Jumlah baris hasil akhir:", df_final.count())

[Stage 47:>                                                         (0 + 3) / 3]

Jumlah baris hasil akhir: 5000


In [25]:
df_final.show(10)

+----------+--------+------------+-------------------+------+------+-----------+----------------+----------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|nama_produk|total_pendapatan|ada_ulasan|  kategori|
+----------+--------+------------+-------------------+------+------+-----------+----------------+----------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Produk-21|           50000|      true|Elektronik|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Produk-25|          100000|      true|Elektronik|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Produk-10|          225000|      true|Elektronik|
|         7|     TX8|           2|2026-10-09 00:00:00|     5| 75000|   Produk-7|          150000|      true|Elektronik|
|        30|     TX9|           1|2026-10-27 00:00:00|     0|250000|  Produk-30|          250000|     false|Elektronik|
|        15|    TX11|           4|2026-1

**E. Insight Akhir**

In [30]:
from pyspark.sql.functions import col, count, sum, round

insight = df_final.groupBy("kategori").agg(
    count("*").alias("total_transaksi"),
    sum(col("ada_ulasan").cast("int")).alias("transaksi_dengan_ulasan")
)

insight = insight.withColumn(
    "persentase_ulasan",
    round(
        col("transaksi_dengan_ulasan") / col("total_transaksi") * 100,
        2
    )
)

insight.show()

[Stage 51:===================>                                      (1 + 2) / 3]

+------------+---------------+-----------------------+-----------------+
|    kategori|total_transaksi|transaksi_dengan_ulasan|persentase_ulasan|
+------------+---------------+-----------------------+-----------------+
|  Elektronik|           1653|                   1168|            70.66|
|     Fashion|           1035|                    726|            70.14|
|Rumah Tangga|            815|                    575|            70.55|
|   Kesehatan|            961|                    662|            68.89|
|     Makanan|            536|                    369|            68.84|
+------------+---------------+-----------------------+-----------------+



**EKSPLORASI**

In [32]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = df_transaksi.join(df_produk, "product_id") \
    .withColumn("pendapatan", F.col("unit_terjual") * F.col("harga"))

w = Window.partitionBy("kategori").orderBy(F.col("pendapatan").desc())

df.withColumn("ranking", F.row_number().over(w)) \
  .filter("ranking = 1") \
  .select("kategori", "nama_produk", "pendapatan") \
  .show(truncate=False)


[Stage 52:>                                                         (0 + 1) / 1]

+------------+-----------+----------+
|kategori    |nama_produk|pendapatan|
+------------+-----------+----------+
|Elektronik  |Produk-2   |1750000   |
|Fashion     |Produk-8   |1750000   |
|Kesehatan   |Produk-11  |700000    |
|Makanan     |Produk-9   |1050000   |
|Rumah Tangga|Produk-29  |1750000   |
+------------+-----------+----------+



In [33]:
from pyspark.sql import functions as F

df_transaksi.join(df_produk, "product_id") \
    .groupBy("product_id", "nama_produk", "kategori") \
    .agg(
        F.sum("unit_terjual").alias("unit_terjual"),
        F.sum(F.col("unit_terjual") * F.col("harga")).alias("pendapatan")
    ) \
    .orderBy(F.col("unit_terjual").desc()) \
    .limit(10) \
    .show(truncate=False)


+----------+-----------+----------+------------+----------+
|product_id|nama_produk|kategori  |unit_terjual|pendapatan|
+----------+-----------+----------+------------+----------+
|12        |Produk-12  |Makanan   |788         |39400000  |
|17        |Produk-17  |Kesehatan |757         |56775000  |
|7         |Produk-7   |Elektronik|753         |56475000  |
|10        |Produk-10  |Elektronik|742         |55650000  |
|18        |Produk-18  |Fashion   |731         |109650000 |
|13        |Produk-13  |Fashion   |715         |178750000 |
|4         |Produk-4   |Makanan   |710         |106500000 |
|21        |Produk-21  |Elektronik|709         |17725000  |
|26        |Produk-26  |Kesehatan |702         |52650000  |
|3         |Produk-3   |Fashion   |696         |52200000  |
+----------+-----------+----------+------------+----------+



In [34]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
